# 1. OUTPUT CONFIGURATION

In [ ]:
import os
from datetime import datetime

def create_output_structure(base="output"):
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    root = f"{base}_{run_id}"

    paths = {
        "root": root,
        "data": f"{root}/data",
        "network": f"{root}/network",
        "topics": f"{root}/topics",
        "metrics": f"{root}/metrics",
        "reports": f"{root}/reports"
    }

    for p in paths.values():
        os.makedirs(p, exist_ok=True)

    return paths

# 2. FULL PIPELINE EXECUTION

In [ ]:
def run_text_analysis(pdf_path):

    paths = create_output_structure()

    # --- LOAD + CLEAN ---
    pages = load_pdf(pdf_path)
    pages = remove_headers_footers(pages)

    # --- CHUNK ---
    chunks = chunk_text(pages)

    # --- EMBEDDINGS + TOPICS ---
    embeddings = embed(chunks)
    topic_model_obj, topics = topic_model(chunks)

    # --- CLUSTER ---
    clusters = cluster(embeddings)

    # --- BUILD DATASET ---
    df = build_dataset(chunks, topics, clusters)

    # --- NETWORK ---
    G = build_network(df)

    # --- TARGETS ---
    target_df = aggregate_targets(df)

    # --- SAVE EVERYTHING ---
    save_all_outputs(df, target_df, G, topic_model_obj, paths)

    print(f"✅ Analysis complete → {paths['root']}")
    return df, G, target_df

# 3. SAVE ALL OUTPUTS

In [ ]:
import json
import networkx as nx

def save_all_outputs(df, targets, G, topic_model_obj, paths):

    # =========================
    # 📊 DATA LAYER
    # =========================
    df.to_parquet(f"{paths['data']}/dataset.parquet")
    df.to_csv(f"{paths['data']}/dataset.csv", index=False)

    targets.to_csv(f"{paths['data']}/targets.csv", index=False)

    # =========================
    # 😊 ENTITY SENTIMENT
    # =========================
    sentiment_rows = []
    for row in df["sentiment"]:
        sentiment_rows.extend(row)

    if sentiment_rows:
        import pandas as pd
        sent_df = pd.DataFrame(sentiment_rows)
        sent_df.to_csv(f"{paths['data']}/entity_sentiment.csv", index=False)

    # =========================
    # 🔗 NETWORK
    # =========================
    edges = nx.to_pandas_edgelist(G)
    edges.to_csv(f"{paths['network']}/network_edges.csv", index=False)

    nx.write_graphml(G, f"{paths['network']}/network.graphml")

    # =========================
    # 🧠 TOPICS
    # =========================
    topic_info = topic_model_obj.get_topic_info()
    topic_info.to_csv(f"{paths['topics']}/topics_summary.csv", index=False)

    topics_json = {
        topic: topic_model_obj.get_topic(topic)
        for topic in topic_model_obj.get_topics().keys()
    }

    with open(f"{paths['topics']}/topics.json", "w") as f:
        json.dump(topics_json, f, indent=2)

    # =========================
    # 📈 FINGERPRINT
    # =========================
    fingerprint = build_fingerprint(df)

    with open(f"{paths['metrics']}/fingerprint.json", "w") as f:
        json.dump(fingerprint, f, indent=2)

    # =========================
    # 📊 SECTION ANALYSIS
    # =========================
    section_df = df.groupby("section").agg({
        "moral": "mean",
        "threat": "mean",
        "power": "mean"
    }).reset_index()

    section_df.to_csv(f"{paths['data']}/sections.csv", index=False)

    try:
        section_df.to_excel(f"{paths['data']}/sections.xlsx", index=False)
    except:
        pass

# 4. AUTO REPORT (DOCX)

In [ ]:
from docx import Document

def generate_report(paths):

    doc = Document()
    doc.add_heading("Text Analysis Report", 0)

    doc.add_heading("Summary", level=1)
    doc.add_paragraph("This report summarizes key findings from the document audit pipeline.")

    doc.add_heading("Outputs", level=1)
    doc.add_paragraph(f"Data stored in: {paths['root']}")

    doc.save(f"{paths['reports']}/report.docx")

# 5. RUN

In [ ]:
df, G, targets = run_text_analysis("your_document.pdf")